# 06 - Foundation Model Comparison (FinBERT-only baseline)

A lightweight foundation-model baseline that predicts forward returns from
**FinBERT outputs alone** (no price features). This isolates how much
predictive signal lives in the news text by itself, in contrast to the
fused 14-feature model from notebook 04.

Optional per the brief - included so the comparison story is complete.


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 110
import warnings; warnings.filterwarnings('ignore', message='enable_nested_tensor.*')

from src import data_loader as dl, train as tr, evaluate as ev
from src.data_loader import (PROJECT_ROOT, MODELS_DIR, PLOTS_DIR, HORIZONS,
                             SEED, SENTIMENT_FEATURES)
from src.train import TrainConfig

MODELS_DIR.mkdir(parents=True, exist_ok=True); PLOTS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Build a sentiment-only sliding-window dataset

In [2]:
ds = dl.assemble_dataset(use_sentiment=True, window=20)
sent_idx = [ds['feature_names'].index(c) for c in SENTIMENT_FEATURES]
X_train = ds['X_train'][:, :, sent_idx]
X_val   = ds['X_val'][:, :, sent_idx]
X_test  = ds['X_test'][:, :, sent_idx]
print('shapes:', X_train.shape, X_val.shape, X_test.shape)


shapes: (4353, 20, 8) (943, 20, 8) (939, 20, 8)


## 2. Tiny transformer over the 8 sentiment features

In [3]:
import torch.nn as nn

class SentimentOnly(nn.Module):
    def __init__(self, n_features=8, seq_len=20, d_model=64, n_heads=2, n_layers=1, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        self.pos = nn.Embedding(seq_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=128,
                                            dropout=dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x):
        b, t, _ = x.shape
        h = self.proj(x) + self.pos(torch.arange(t, device=x.device).unsqueeze(0).expand(b, t))
        h = self.enc(h)
        return self.head(h[:, -1])

tr.set_seed(SEED)
model = SentimentOnly()
print('params:', sum(p.numel() for p in model.parameters()))


params: 37507


## 3. Train and evaluate

In [4]:
cfg = TrainConfig(epochs=40, batch_size=64, lr=5e-4,
                  weight_decay=1e-5, grad_clip=1.0, early_stopping_patience=6, seed=SEED)
log_rows = []
def on_epoch(epoch, tloss, vloss, dt, improved):
    flag = '<-- best' if improved else ''
    print(f'epoch {epoch:>3} | train MSE {tloss:.6f} | val MSE {vloss:.6f} | {dt:5.1f}s {flag}')
    log_rows.append({'epoch':epoch,'train_loss':tloss,'val_loss':vloss,'seconds':dt})

save_path = MODELS_DIR / 'foundation_sentiment_only.pt'
history = tr.train_model(model, X_train, ds['y_train'], X_val, ds['y_val'],
                         cfg, save_path, on_epoch_end=on_epoch)
print('best val MSE:', f"{history.best_val_loss:.6f}")


epoch   1 | train MSE 0.007460 | val MSE 0.000734 |   0.7s <-- best


epoch   2 | train MSE 0.001156 | val MSE 0.000633 |   0.7s <-- best


epoch   3 | train MSE 0.001044 | val MSE 0.000611 |   0.7s <-- best


epoch   4 | train MSE 0.001007 | val MSE 0.000587 |   0.7s <-- best


epoch   5 | train MSE 0.000940 | val MSE 0.000614 |   0.9s 


...

epoch  15 | train MSE 0.000867 | val MSE 0.000586 |   0.8s 


epoch  16 | train MSE 0.000855 | val MSE 0.000577 |   0.8s 


epoch  17 | train MSE 0.000842 | val MSE 0.000589 |   0.9s 


epoch  18 | train MSE 0.000841 | val MSE 0.000575 |   0.8s 


epoch  19 | train MSE 0.000842 | val MSE 0.000601 |   0.9s 
best val MSE: 0.000573


In [5]:
yhat = tr.predict(model, X_test)
mtr = ev.per_horizon_metrics(ds['y_test'], yhat, HORIZONS, 'sentiment_only')
mtr.to_csv(PROJECT_ROOT / 'results' / 'sentiment_only_metrics.csv', index=False)
mtr


,model,horizon,mse,mae,directional_accuracy,n
0,sentiment_only,t+1,0.000220,0.011378,0.494143,939
1,sentiment_only,t+3,0.000593,0.018610,0.518637,939
2,sentiment_only,t+5,0.000918,0.023200,0.582535,939


**Interpretation:** if `sentiment_only` directional accuracy is well below
chance, the FinBERT signal alone is too weak to beat random - meaning the
gain (if any) in notebook 05 comes from the **interaction** between sentiment
and price, not sentiment as a standalone signal.
